# Huấn luyện Thử nghiệm 1 Epoch với Multi-GPU DDP và AMP trên Kaggle

Notebook này được thiết kế để chạy huấn luyện mô hình Transformer Seq2Seq trên **toàn bộ bộ dữ liệu** (Full Dataset) trong **1 Epoch** nhằm kiểm chứng tính đúng đắn của:
- **Distributed Data Parallel (DDP)** trên 2 GPU Tesla T4 của Kaggle.
- **Automatic Mixed Precision (AMP)** cho tối ưu bộ nhớ và thời gian tính toán.
- **Learning Rate Scheduler** hoạt động chính xác (Cosine Decay + Warmup).
- **Loss không bị NaN** và hội tụ mượt mà.
- **Validation Loss & Perplexity (PPL)** tính toán và đồng bộ chính xác.
- **Checkpoint Save/Load** hoạt động chuẩn xác.

## Cấu hình chạy thử nghiệm:
- Số epochs: `1`
- Early Stopping: Tắt (`enabled: false`)
- Validation: Chỉ tính `val_loss` và `perplexity` (tắt ROUGE để tiết kiệm thời gian)
- Decoding Strategy: Sử dụng `greedy` để kiểm thử sinh văn bản nhanh
- Các tham số khác bám sát file `configs/transformer_summarization.yaml`.

In [ ]:
# Copy thư mục src và configs từ Input sang Working directory
!cp -r /kaggle/input/datasets/tranducthinh2006/vietnamese-transformer-summarization-data/data/src .
!cp -r /kaggle/input/datasets/tranducthinh2006/vietnamese-transformer-summarization-data/data/configs .

# Tạo liên kết mềm (symlink) cho dữ liệu
!mkdir -p data
!ln -s /kaggle/input/datasets/tranducthinh2006/vietnamese-transformer-summarization-data/data/bin data/bin
!ln -s /kaggle/input/datasets/tranducthinh2006/vietnamese-transformer-summarization-data/data/tokenizer data/tokenizer


In [ ]:
# 1. Cài đặt các thư viện bổ sung nếu chạy trên Kaggle
!pip install -q sentencepiece pyyaml torch tqdm

In [1]:
import os
import sys
import torch

# Thêm thư mục dự án vào python path để import src
if os.path.exists("../src"):
    sys.path.append(os.path.abspath(".."))
    print("Đã kết nối với thư mục dự án cục bộ (Local Parent Directory).")
elif os.path.exists("./vietnamese_transformer_summarization"):
    sys.path.append(os.path.abspath("./vietnamese_transformer_summarization"))
    print("Đã kết nối với thư mục dự án trên Kaggle.")

## Kiểm tra Tài nguyên phần cứng GPU

In [2]:
print("=== THÔNG TIN PHẦN CỨNG GPU ===")
print(f"CUDA khả dụng: {torch.cuda.is_available()}")
print(f"Số lượng GPU khả dụng: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

=== THÔNG TIN PHẦN CỨNG GPU ===
CUDA khả dụng: True
Số lượng GPU khả dụng: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


## Kích hoạt huấn luyện Multi-GPU DDP bằng `torchrun`

Chúng ta sử dụng `torchrun` để tự động thiết lập các biến môi trường cần thiết cho DDP (như `WORLD_SIZE`, `LOCAL_RANK`, `MASTER_ADDR`, `MASTER_PORT`) và khởi chạy 2 tiến trình song song tương ứng với 2 GPU Tesla T4 trên Kaggle.

Cờ `--verify_1epoch` sẽ tự động ghi đè cấu hình huấn luyện sang chế độ kiểm chứng 1 epoch và cấu hình tinh giản (greedy decoding, no early stopping, no ROUGE metrics) giúp quá trình kiểm chứng diễn ra nhanh nhất.

In [3]:
# Khởi chạy song song trên 2 GPU sử dụng torchrun
# Nếu chạy trên môi trường chỉ có 1 GPU, có thể sửa --nproc_per_node=1
!torchrun --nproc_per_node=2 src/train.py \
    --config configs/transformer_summarization.yaml \
    --verify_1epoch

W0608 09:08:13.160000 152 torch/distributed/run.py:852] 
W0608 09:08:13.160000 152 torch/distributed/run.py:852] *****************************************
W0608 09:08:13.160000 152 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0608 09:08:13.160000 152 torch/distributed/run.py:852] *****************************************
[2026-06-08 09:08:15] INFO [TrainPipeline:84] Khởi chạy tiến trình. Rank: 0, World Size: 2, Device: cuda:0, DDP: True
[2026-06-08 09:08:15] INFO [TrainPipeline:89] Đang áp dụng ghi đè cấu hình thử nghiệm 1 epoch trên Kaggle...
[2026-06-08 09:08:15] INFO [TrainPipeline:100] Đang tải Tokenizer từ: data/tokenizer/vietnamese_spm.model
[2026-06-08 09:08:15] INFO [TrainPipeline:103] Đang nạp dataset huấn luyện từ: data/bin/train_token_id.jsonl
[2026-06-08 09:08:39] INFO [TrainPipeli

## Phân tích kết quả sau 1 Epoch

Sau khi lệnh chạy hoàn thành, vui lòng quan sát dòng log đầu ra từ tiến trình chính (Rank 0) để kiểm tra các tiêu chí:
1. **DDP & AMP**: Cả hai GPU hoạt động song song, không lỗi tràn bộ nhớ (OOM).
2. **Loss không bị NaN**: Các giá trị loss hiển thị giảm dần đều đặn qua từng bước log.
3. **Train Loss & Val Loss**: Loss kết thúc Epoch và Loss trên tập Validation đều được tính toán và log rõ ràng.
4. **Perplexity (PPL)**: Giá trị Perplexity hiển thị (mục tiêu là giảm dần cùng với loss).
5. **Greedy Generation (20 mẫu)**: Dưới chân màn hình hiển thị 20 cặp câu `Reference` (Tóm tắt gốc) và `Generated` (Mô hình sinh) để kiểm tra chất lượng tóm tắt.
6. **GPU Memory & Time per Epoch**: Log in ra lượng VRAM đỉnh (MB) đã sử dụng và tổng thời gian hoàn thành epoch (giây).

In [4]:
!zip -j checkpoint.zip checkpoints/transformer_base/last.pt


  adding: last.pt (deflated 8%)


In [5]:
from IPython.display import FileLink
FileLink('checkpoint.zip')


/kaggle/working/checkpoint.zip